# 🧠 OO-Native Model — Training Local (VS Code)

Ce notebook tourne **localement** dans VS Code (pas besoin de Google Drive).

⚠️ **Sans GPU NVIDIA** → utilise CPU uniquement → dry-run OK, full training lent.

Pour le full training avec GPU → utilise [Google Colab](https://colab.research.google.com) avec le fichier `train_colab.ipynb`.

In [1]:
# ── 1. Setup chemin ──────────────────────────────────────────────────
import os, sys

# Chemin vers le dossier oo-model (automatique si le notebook est dans notebooks/)
OO_MODEL_PATH = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
# Ou forcer le chemin absolu:
# OO_MODEL_PATH = r'C:\Users\djibi\OneDrive\Bureau\baremetal\oo-model'

os.chdir(OO_MODEL_PATH)
if 'src' not in sys.path:
    sys.path.insert(0, os.path.join(OO_MODEL_PATH, 'src'))

print('CWD:', os.getcwd())
print('Python:', sys.version)

CWD: c:\Users\djibi\OneDrive\Bureau\baremetal\oo-model
Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]


In [2]:
# ── 2. Vérification environnement ────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Mode CPU — dry-run OK, full training lent (~heures)')
    print('Pour GPU gratuit: https://colab.research.google.com')

PyTorch: 2.9.1+cpu
CUDA disponible: False
Mode CPU — dry-run OK, full training lent (~heures)
Pour GPU gratuit: https://colab.research.google.com


In [3]:
# ── 3. Construire le dataset OO ──────────────────────────────────────
import subprocess, json

result = subprocess.run([sys.executable, 'scripts/build_dataset.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERREUR:', result.stderr)

with open('data/processed/train.jsonl', encoding='utf-8') as f:
    rows = [json.loads(l) for l in f]
print(f'Dataset: {len(rows)} samples train')
print('Exemple:', rows[0]['instruction'][:80])

[dataset] data\processed\train.jsonl: 53 rows  (domains: {'chat', 'system', 'math', 'code', 'policy', 'arch'})
[dataset] data\processed\valid.jsonl: 6 rows  (domains: {'chat', 'system', 'arch', 'code'})
[dataset] data\processed\test.jsonl: 8 rows  (domains: {'chat', 'system', 'arch', 'code', 'math'})
[dataset] data\processed\eval_oo.jsonl: 34 rows  (domains: {'policy', 'arch', 'system'})

[dataset] Total: 67 samples
          arch: 10
          chat: 9
          code: 12
          math: 12
          policy: 9
          system: 15

Dataset: 53 samples train
Exemple: What is D+ (D-Plus)?


In [5]:
# ── 4. Dry-run (20 steps, rapide sur CPU) ────────────────────────────
# Valide que l'architecture fonctionne correctement
import subprocess

result = subprocess.run(
    [sys.executable, 'scripts/train_oo_native.py', 'configs/oo_native_v1.json', '--dry-run'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('ERREUR:', result.stderr[-2000:])

[oo-native] device=cpu dry_run=True
[oo-native] Tokenizer vocab_size=833
[oo-native] Model params: 4,594,082 total | 4,594,082 trainable
[oo-native] Dataset: 53 samples
[oo-native] step=1/20 loss=6.3283 lr=0.000001
[oo-native] step=2/20 loss=6.3482 lr=0.000003
[oo-native] step=3/20 loss=6.3063 lr=0.000004
[oo-native] step=4/20 loss=6.0779 lr=0.000006
[oo-native] step=5/20 loss=6.2833 lr=0.000007
[oo-native] step=6/20 loss=6.1404 lr=0.000009
[oo-native] step=7/20 loss=6.2797 lr=0.000010
[oo-native] step=8/20 loss=5.8344 lr=0.000012
[oo-native] step=9/20 loss=6.2684 lr=0.000013
[oo-native] step=10/20 loss=5.8257 lr=0.000015
[oo-native] step=11/20 loss=5.9806 lr=0.000016
[oo-native] step=12/20 loss=6.0297 lr=0.000018
[oo-native] step=13/20 loss=5.8509 lr=0.000019
[oo-native] step=14/20 loss=5.8405 lr=0.000021
[oo-native] step=15/20 loss=5.6937 lr=0.000022
[oo-native] step=16/20 loss=5.7994 lr=0.000024
[oo-native] step=17/20 loss=5.7356 lr=0.000025
[oo-native] step=18/20 loss=5.5377 lr=0.0

In [6]:
# ── 5. Full training (CPU, lent) ─────────────────────────────────────
# ⚠️ Sur CPU: ~2-4 heures pour 5000 steps
# Sur Colab T4: ~25 minutes
# Décommenter pour lancer:

# result = subprocess.run(
#     [sys.executable, 'scripts/train_oo_native.py', 'configs/oo_native_v1.json'],
#     capture_output=False  # affiche en temps réel
# )

print('Full training commenté — décommenter la cellule pour lancer')
print('Recommandation: utiliser Colab T4 (25min) plutôt que CPU local (2-4h)')

Full training commenté — décommenter la cellule pour lancer
Recommandation: utiliser Colab T4 (25min) plutôt que CPU local (2-4h)


In [7]:
# ── 6. Charger et tester le checkpoint ───────────────────────────────
import torch
from pathlib import Path
from oo_model.oo_native import OONativeModel, OONativeConfig
from oo_model.oo_tokenizer import OOTokenizer

CKPT = Path('checkpoints/oo-native-v1') / 'oo_native_v1.pt'
TOK_PATH = Path('checkpoints/oo-native-v1') / 'tokenizer.json'

if not CKPT.exists():
    print('Pas de checkpoint — lance le dry-run ou le full training d abord')
else:
    state = torch.load(CKPT, map_location='cpu', weights_only=False)
    saved_cfg = state.get('config', {})
    model_cfg = OONativeConfig(**saved_cfg) if saved_cfg else OONativeConfig()
    model = OONativeModel(model_cfg)
    model_state = state.get('model_state') or state.get('model_state_dict')
    model.load_state_dict(model_state)
    model.eval()

    tok = OOTokenizer.load(str(TOK_PATH)) if TOK_PATH.exists() else None

    params = sum(p.numel() for p in model.parameters())
    print(f'Modele charge: {params/1e6:.2f}M params')
    print(f'Couches: {model.cfg.n_layer}')
    if tok is not None:
        print(f'Tokenizer: {len(tok.vocab)} tokens')
    else:
        print('Tokenizer introuvable')
    print('Checkpoint charge avec succes ✓')

Modele charge: 4.59M params
Couches: 2
Tokenizer: 833 tokens
Checkpoint charge avec succes ✓
